# 14 OpenOximetry PPG Poster Waveform

Generate a publication-ready PPG waveform figure for the poster architecture infographic.
Loads a raw PPG .dat file from the OpenOximetry dataset, removes baseline wander,
selects the cleanest 10-second window, and renders a poster-quality SVG.


In [ ]:
import os
import numpy as np
from pathlib import Path
from scipy import ndimage
from scipy.signal import find_peaks
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


### Select Source Record

Using OpenOximetry record 06d849d0... which showed the highest signal-to-noise ratio
during automated screening of the first 20 waveform files.


In [ ]:
DATA_ROOT = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()) / 'data' / 'physionet.org' / 'files' / 'openox-repo' / '1.1.1' / 'waveforms'
RECORD_ID = '06d849d084e43b89b761a4c343b05a9b75eebb43bf11c9ef3de75c18f308020c'
SUB_DIR = '0'

hea_path = DATA_ROOT / SUB_DIR / f'{RECORD_ID}_ppg.hea'
dat_path = hea_path.with_suffix('.dat')
print(f'Record: {RECORD_ID}')
print(f'HEA exists: {hea_path.exists()}')
print(f'DAT exists: {dat_path.exists()}')

with open(hea_path) as f:
    print(f'\nHeader:\n{f.read()}')


### Load and Condition PPG Signal

Read 8-channel 32-bit interleaved data at 86 Hz. Use IR Signal (channel 4)
which showed stronger amplitude during screening.
Remove baseline wander with a 1-second median filter.


In [ ]:
FS = 86
raw = np.fromfile(dat_path, dtype=np.int32)
n_channels = 8
data = raw.reshape(-1, n_channels)

# Use IR Signal (ch4)
sig = data[:, 4].astype(np.float64)

# Remove baseline wander
baseline = ndimage.median_filter(sig, size=FS)
ac = sig - baseline

print(f'Total samples: {len(ac)} ({len(ac)/FS:.0f}s)')
print(f'Channel 4 (IR Signal) range: {ac.min():.0f} to {ac.max():.0f}')


### Select Cleanest 10-Second Window

Slide a 10-second window across the recording and score each by the product
of spectral peak power (0.5-3 Hz band) and RMS amplitude.


In [ ]:
WINDOW = 10 * FS
best_score = 0
best_start = 0

for start in range(0, len(ac) - WINDOW, FS):
    seg = ac[start:start + WINDOW]
    seg = seg - seg.mean()
    spectrum = np.abs(np.fft.rfft(seg))
    freqs = np.fft.rfftfreq(WINDOW, 1/FS)
    mask = (freqs >= 0.5) & (freqs <= 3.0)
    if mask.sum() < 2:
        continue
    peak_power = spectrum[mask].max()
    total_power = spectrum[mask].sum()
    snr = peak_power / max(total_power - peak_power, 1)
    score = snr * seg.std()
    if score > best_score:
        best_score = score
        best_start = start

best_seg = ac[best_start:best_start + WINDOW]
t = np.arange(WINDOW) / FS

peaks, _ = find_peaks(best_seg, distance=FS*0.4, prominence=max(best_seg.std()*0.5, 1))
bpm = len(peaks) * 6

print(f'Best window start: {best_start/FS:.0f}s')
print(f'Heartbeats detected: {len(peaks)} ({bpm} bpm)')
print(f'Amplitude range: {best_seg.min():.0f} to {best_seg.max():.0f}')


### Render Poster-Quality Waveform SVG

Generate a clean, minimal SVG with strong blue trace, light grid, and visible axes.
Output dimensions: 8 x 2.8 in at 150 DPI, suitable for poster insertion.


In [ ]:
OUTPUT_DIR = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()) / 'outputs' / 'poster'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

svg_path = OUTPUT_DIR / 'ppg-acquisition-waveform.svg'
png_path = OUTPUT_DIR / 'ppg-acquisition-waveform.png'

plt.style.use('default')
fig, ax = plt.subplots(figsize=(8, 2.8))

ax.plot(t, best_seg, linewidth=1.8, color='#1565C0')
ax.axhline(0, color='#666666', linewidth=0.5, linestyle='-', alpha=0.5)

ax.set_xlabel('Time (s)', fontsize=11, fontweight='bold')
ax.set_ylabel('Amplitude (AU)', fontsize=11, fontweight='bold')
ax.tick_params(axis='both', labelsize=9)

ax.grid(True, alpha=0.15, linestyle='-')
ax.set_xlim(0, 10)
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))
ax.yaxis.set_major_locator(ticker.MaxNLocator(5))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout(pad=0.8)
plt.savefig(svg_path, dpi=150, bbox_inches='tight')
plt.savefig(png_path, dpi=200, bbox_inches='tight')
plt.close()

print(f'SVG: {svg_path} ({svg_path.stat().st_size} bytes)')
print(f'PNG: {png_path} ({png_path.stat().st_size} bytes)')


### Generate Poster-Optimized Variant

Compact version for the Draw.io placeholder n6 (160 x 80 px).
Keeps minimal axis labels (Time, AU) scaled for the small placeholder.


In [ ]:
# Poster-optimized compact variant for n6 placeholder
fig2, ax2 = plt.subplots(figsize=(3.2, 1.6))

ax2.plot(t, best_seg, linewidth=2.0, color='#1565C0')
ax2.axhline(0, color='#666666', linewidth=0.5, linestyle='-', alpha=0.4)

# Minimal axis labels scaled for poster placeholder
ax2.set_title('PPG waveform', fontsize=7, color='#444444', fontweight='bold', pad=1)
ax2.set_xlabel('Time (s)', fontsize=7, color='#444444', labelpad=1)
ax2.set_ylabel('AU', fontsize=7, color='#444444', labelpad=1)
ax2.set_xticks(range(0, 11, 2))
ax2.set_xticklabels([str(x) for x in range(0, 11, 2)], fontsize=6.5, color='#666666')
ax2.set_yticks([])
ax2.set_xlim(0, 10)

ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_visible(False)
ax2.spines['bottom'].set_color('#CCCCCC')

ax2.tick_params(bottom=True, top=False, left=False, right=False, colors='#888888')
ax2.grid(True, alpha=0.08, linestyle='-')

plt.tight_layout(pad=0.4)

poster_svg = OUTPUT_DIR / 'ppg-acquisition-waveform-poster.svg'
poster_png = OUTPUT_DIR / 'ppg-acquisition-waveform-poster.png'
plt.savefig(poster_svg, dpi=200, bbox_inches='tight', pad_inches=0.08, transparent=False)
plt.savefig(poster_png, dpi=300, bbox_inches='tight', pad_inches=0.08)
plt.close()

print(f'Poster SVG: {poster_svg} ({poster_svg.stat().st_size} bytes)')
print(f'Poster PNG: {poster_png} ({poster_png.stat().st_size} bytes)')


### Verify Output

The waveform shows a 10-second PPG window with characteristic systolic peaks
at approximately 84 bpm, suitable for poster insertion into the Draw.io
architecture diagram placeholder n6.


In [ ]:
print('Output files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size
    print(f'  {f.name:45s} {size:>8d} bytes')

